In [10]:
# !pip install docplex
# !pip install memory_profiler
# !pip install dwave-system
# !pip install gurobipy
# !pip install yfinance

In [2]:
###############################################################################
# // SPDX-License-Identifier: Apache-2.0
# // Copyright 2024: Amazon Web Services, Inc. - Contributions from JPMC
###############################################################################

import numpy as np
import pandas as pd
from itertools import product
import sys
sys.path.append("../")
from dcmppln.pipeline import Pipeline
from dcmppln.optimizer import GurobiOptimizer
from tests.input_data import InputData
from dcmppln.optimizer import SimulatedAnnealingDwave
import yfinance as yf
from typing import List

In [53]:
def download_data(stocks: List[str], start_date: str, end_date: str, period_step: int) -> pd.DataFrame:
    stock_data = {}
    for stock in stocks:
        ticker = yf.Ticker(stock)
        stock_data[stock] = ticker.history(start=start_date, end=end_date)["Close"][::period_step]
    return pd.DataFrame(stock_data)

def calculate_return(data: pd.DataFrame) -> pd.DataFrame:
    log_return = np.log(data / data.shift(1))
    return log_return.iloc[1:]  # Drop first row as it will have NaN

# List of stock markets
markets = {
    #"payment_processors": ["V", "MA", "PYPL", "AXP", "DFS", "FI", "GPN", "FIS", "SQ", "ADP"],
    #"defense_contractors": ["LMT", "RTX", "NOC", "GD", "BA", "HII", "TXT", "LDOS", "CW", "LHX"],
    "consumer_staples": ["PG", "KO", "PEP", "WMT", "COST", "CL", "MDLZ", "KHC", "TGT", "SYY"],
    #"cloud_computing": ["MSFT", "AMZN", "GOOGL", "CRM", "ORCL", "IBM", "NOW", "DDOG", "ZS", "PANW"],
    #"autonomous_vehicles": ["TSLA", "GM", "F", "GOOGL", "NVDA", "INTC", "MBLY", "APTIV", "TM", "LI"],
    #"ecommerce_giants": ["AMZN", "BABA", "SHOP", "WMT", "EBAY", "MELI", "JD", "SE", "PDD", "TGT"],
    #"biotech_innovators": ["BIIB", "REGN", "VRTX", "ILMN", "CRSP", "NTLA", "BEAM", "EDIT", "BMRN", "XBI"],
    #"metaverse_stocks": ["META", "RBLX", "U", "NVDA", "MSFT", "GOOGL", "DIS", "SONY", "AAPL", "MTTR"],
    #"cybersecurity_leaders": ["PANW", "FTNT", "CRWD", "ZS", "OKTA", "CHKP", "MSFT", "NET", "CYBR", "BUG"],
    "semiconductor_giants": ["NVDA", "AMD", "INTC", "TSM", "QCOM", "AVGO", "ASML", "TXN", "MU", "LRCX"],
    #"green_energy": ["TSLA", "PLUG", "ENPH", "SEDG", "FSLR", "RUN", "NEE", "BE", "BLDP", "ICLN"],
    #"space_exploration": ["SPCE", "RKLB", "BA", "LMT", "NOC", "IRDM", "AJRD", "MAXR", "AVAV", "UFO"],
    "tech_titans": ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "QQQ", "VGT", "IGM"],
    "financial_giants": ["JPM", "BAC", "GS", "WFC", "C", "MS", "SCHW", "V", "MA", "XLF"],
    "retail_consumer": ["WMT", "TGT", "COST", "HD", "LOW", "ULTA", "NKE", "SBUX", "AMZN", "XRT"],
    "healthcare_leaders": ["UNH", "JNJ", "PFE", "MRK", "ABBV", "LLY", "BMY", "TMO", "ISRG", "XLV"],
    #"energy_utilities": ["XOM", "CVX", "BP", "OXY", "COP", "ENB", "KMI", "NEE", "DUK", "XLE"],
    "industrial_powerhouses": ["CAT", "GE", "HON", "DE", "LMT", "BA", "MMM", "RTX", "EMR", "XLI"],
    #"real_estate_reits": ["VNQ", "O", "SPG", "AMT", "CCI", "PSA", "VICI", "PLD", "WELL", "XLRE"],
    #"entertainment_media": ["DIS", "NFLX", "CMCSA", "WBD", "PARA", "SPOT", "EA", "ROKU", "FOXA", "XLC"],  # Replaced SONY with EA
    #"international_giants": ["BABA", "TSM", "ASML", "NVS", "SAP", "TM", "LVMUY", "RIO", "VALE", "IXUS"],
    "defensive_plays": ["KO", "PG", "JNJ", "MCD", "PEP", "WMT", "XLU", "VYM", "TLT", "VGIT"],
    }

# Define time period for analysis
start_date = "2010-07-01"
end_date = "2020-07-01"
period_step = 5  # Weekly changes

# Initialize dataframe to store results
results_df = pd.DataFrame(columns=["Distribution", "Market", "Optimizer", "Correlations", "Covariances", "Returns", "DCT"])

### 2a: MP Distribution

In [54]:
# Loop through each market and analyze the stocks
# 2a Run:
for market_name, stocks in markets.items():
    print(f"\nProcessing market: {market_name}")
    
    # Distribution Type:
    distribution = "MP Distribution"
    
    # Download stock price data
    dataset = download_data(stocks, start_date, end_date, period_step)

    # Compute log returns
    log_period_returns = calculate_return(dataset)

    # Compute average returns for each stock (mean of log returns)
    returns = log_period_returns.mean().to_numpy()

    # Compute covariance matrix of log returns
    covariances = log_period_returns.cov().to_numpy()

    # Compute correlation matrix of log returns
    correlations = log_period_returns.corr().to_numpy()

    # Print output shapes for verification
    print("Returns shape:", returns.shape)  # (10,)
    print("Covariance matrix shape:", covariances.shape)  # (10, 10)
    print("Correlation matrix shape:", correlations.shape)  # (10, 10)")

    # No Optimizer Run
    p = Pipeline(
        correlations,
        covariances,
        returns,
    )
    dct = p.run()

    # Add results to dataframe
    results_df = pd.concat([results_df, pd.DataFrame([[distribution, market_name, "None", correlations, covariances, returns, dct]], 
                                                      columns=results_df.columns)], ignore_index=True)

    # Run with Gurobi Optimizer
    p = Pipeline(
        correlations,
        covariances,
        returns, 
        optimize_func=GurobiOptimizer()
    )
    dct = p.run(run_optimizer=True)

    # Add results to dataframe
    results_df = pd.concat([results_df, pd.DataFrame([[distribution, market_name, "Gurobi", correlations, covariances, returns, dct]], 
                                                      columns=results_df.columns)], ignore_index=True)

    # Run with D-Wave Simulated Annealing
    p = Pipeline(
        correlations,
        covariances,
        returns, 
        optimize_func=SimulatedAnnealingDwave()
    )
    dct = p.run(run_optimizer=True)

    # Add results to dataframe
    results_df = pd.concat([results_df, pd.DataFrame([[distribution, market_name, "Dwave", correlations, covariances, returns, dct]], 
                                                      columns=results_df.columns)], ignore_index=True)


Processing market: consumer_staples
Returns shape: (10,)
Covariance matrix shape: (10, 10)
Correlation matrix shape: (10, 10)
Read LP format model from file temp.lp
Reading time = 0.00 seconds
obj: 1 rows, 10 columns, 10 nonzeros
Set parameter TimeLimit to value 300
Set parameter Threads to value 1
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1185G7 @ 3.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 1 threads

Optimize a model with 1 rows, 10 columns and 10 nonzeros
Model fingerprint: 0x0584b323
Model has 55 quadratic objective terms
Variable types: 0 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-03, 4e-03]
  QObjective range [6e-04, 4e-03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 5e+00]
Found heuristic solution: objective -0.0032659
Presolve time: 0.00s

In [55]:
# Display the results dataframe
results_df

,Distribution,Market,Optimizer,Correlations,Covariances,Returns,DCT
0,MP Distribution,consumer_staples,None,"[[1.0, 0.47386620792483153, 0.6316363086512147...","[[0.0005528967935663235, 0.0002661030132532858...","[0.0019756254266453262, 0.0017536758753608581,...","([[1.0, 0.47386620792483153, 0.631636308651214..."
1,MP Distribution,consumer_staples,Gurobi,"[[1.0, 0.47386620792483153, 0.6316363086512147...","[[0.0005528967935663235, 0.0002661030132532858...","[0.0019756254266453262, 0.0017536758753608581,...","{'score': -0.003418789995558764, 'recombined_s..."
2,MP Distribution,consumer_staples,Dwave,"[[1.0, 0.47386620792483153, 0.6316363086512147...","[[0.0005528967935663235, 0.0002661030132532858...","[0.0019756254266453262, 0.0017536758753608581,...","{'score': -0.00017554955773673234, 'recombined..."
3,MP Distribution,semiconductor_giants,None,"[[1.0, 0.4722724555670802, 0.4972459461894452,...","[[0.003542835578726633, 0.0023177682025648147,...","[0.007259697150247052, 0.0038120867645143786, ...","([[1.0, 0.4722724555670802, 0.4972459461894452..."
4,MP Distribution,semiconductor_giants,Gurobi,"[[1.0, 0.4722724555670802, 0.4972459461894452,...","[[0.003542835578726633, 0.0023177682025648147,...","[0.007259697150247052, 0.0038120867645143786, ...","{'score': 0.003657846920176476, 'recombined_so..."
5,MP Distribution,semiconductor_giants,Dwave,"[[1.0, 0.4722724555670802, 0.4972459461894452,...","[[0.003542835578726633, 0.0023177682025648147,...","[0.007259697150247052, 0.0038120867645143786, ...","{'score': 0.01188873279745787, 'recombined_sol..."
6,MP Distribution,tech_titans,None,"[[1.0, 0.48763133498554195, 0.4455451713649541...","[[0.0014456847012178638, 0.0006075878078786981...","[0.004902196313674589, 0.004739169450291489, 0...","([[1.0, 0.48763133498554195, 0.445545171364954..."
7,MP Distribution,tech_titans,Gurobi,"[[1.0, 0.48763133498554195, 0.4455451713649541...","[[0.0014456847012178638, 0.0006075878078786981...","[0.004902196313674589, 0.004739169450291489, 0...","{'score': -0.002617401998283291, 'recombined_s..."
8,MP Distribution,tech_titans,Dwave,"[[1.0, 0.48763133498554195, 0.4455451713649541...","[[0.0014456847012178638, 0.0006075878078786981...","[0.004902196313674589, 0.004739169450291489, 0...","{'score': 0.0009563896267798175, 'recombined_s..."
9,MP Distribution,financial_giants,None,"[[1.0, 0.8632613381456855, 0.8676079586391919,...","[[0.001576439094087429, 0.0017417705303164324,...","[0.0023889045533935496, 0.0012539891378432974,...","([[1.0, 0.8632613381456855, 0.8676079586391919..."


### 2b: Shrinkage Estimator:

In [56]:
# Redoing imports with changed code:
from dcmppln.pipeline import Pipeline
from dcmppln.optimizer import GurobiOptimizer
from tests.input_data import InputData
from dcmppln.optimizer import SimulatedAnnealingDwave

In [57]:
# Loop through each market and analyze the stocks
# 2b Run:
for market_name, stocks in markets.items():
    print(f"\nProcessing market: {market_name}")
    
    # Distribution Type:
    distribution = "Shrinkage Estimator"
    
    # Download stock price data
    dataset = download_data(stocks, start_date, end_date, period_step)

    # Compute log returns
    log_period_returns = calculate_return(dataset)

    # Compute average returns for each stock (mean of log returns)
    returns = log_period_returns.mean().to_numpy()

    # Compute covariance matrix of log returns
    covariances = log_period_returns.cov().to_numpy()

    # Compute correlation matrix of log returns
    correlations = log_period_returns.corr().to_numpy()

    # Print output shapes for verification
    print("Returns shape:", returns.shape)  # (10,)
    print("Covariance matrix shape:", covariances.shape)  # (10, 10)
    print("Correlation matrix shape:", correlations.shape)  # (10, 10)")

    # No Optimizer Run
    p = Pipeline(
        correlations,
        covariances,
        returns,
    )
    dct = p.run()

    # Add results to dataframe
    results_df = pd.concat([results_df, pd.DataFrame([[distribution, market_name, "None", correlations, covariances, returns, dct]], 
                                                      columns=results_df.columns)], ignore_index=True)

    # Run with Gurobi Optimizer
    p = Pipeline(
        correlations,
        covariances,
        returns, 
        optimize_func=GurobiOptimizer()
    )
    dct = p.run(run_optimizer=True)

    # Add results to dataframe
    results_df = pd.concat([results_df, pd.DataFrame([[distribution, market_name, "Gurobi", correlations, covariances, returns, dct]], 
                                                      columns=results_df.columns)], ignore_index=True)

    # Run with D-Wave Simulated Annealing
    p = Pipeline(
        correlations,
        covariances,
        returns, 
        optimize_func=SimulatedAnnealingDwave()
    )
    dct = p.run(run_optimizer=True)

    # Add results to dataframe
    results_df = pd.concat([results_df, pd.DataFrame([[distribution, market_name, "Dwave", correlations, covariances, returns, dct]], 
                                                      columns=results_df.columns)], ignore_index=True)


Processing market: consumer_staples
Returns shape: (10,)
Covariance matrix shape: (10, 10)
Correlation matrix shape: (10, 10)
Read LP format model from file temp.lp
Reading time = 0.00 seconds
obj: 1 rows, 10 columns, 10 nonzeros
Set parameter TimeLimit to value 300
Set parameter Threads to value 1
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1185G7 @ 3.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 1 threads

Optimize a model with 1 rows, 10 columns and 10 nonzeros
Model fingerprint: 0x08c5768f
Model has 55 quadratic objective terms
Variable types: 0 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-03, 4e-03]
  QObjective range [6e-04, 4e-03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 5e+00]
Found heuristic solution: objective -0.0032659
Presolve time: 0.00s

In [59]:
# Display the results dataframe
results_df = results_df.sort_values(by=["Market", "Optimizer"]).reset_index(drop=True)
results_df

,Distribution,Market,Optimizer,Correlations,Covariances,Returns,DCT
0,MP Distribution,consumer_staples,Dwave,"[[1.0, 0.47386620792483153, 0.6316363086512147...","[[0.0005528967935663235, 0.0002661030132532858...","[0.0019756254266453262, 0.0017536758753608581,...","{'score': -0.00017554955773673234, 'recombined..."
1,Shrinkage Estimator,consumer_staples,Dwave,"[[1.0, 0.473865960659252, 0.6316366670257212, ...","[[0.0005528972424157462, 0.0002661029824125156...","[0.0019756257672612805, 0.0017536758753608581,...","{'score': -0.0019427793804989574, 'recombined_..."
2,MP Distribution,consumer_staples,Gurobi,"[[1.0, 0.47386620792483153, 0.6316363086512147...","[[0.0005528967935663235, 0.0002661030132532858...","[0.0019756254266453262, 0.0017536758753608581,...","{'score': -0.003418789995558764, 'recombined_s..."
3,Shrinkage Estimator,consumer_staples,Gurobi,"[[1.0, 0.473865960659252, 0.6316366670257212, ...","[[0.0005528972424157462, 0.0002661029824125156...","[0.0019756257672612805, 0.0017536758753608581,...","{'score': -0.0034187839442253176, 'recombined_..."
4,MP Distribution,consumer_staples,None,"[[1.0, 0.47386620792483153, 0.6316363086512147...","[[0.0005528967935663235, 0.0002661030132532858...","[0.0019756254266453262, 0.0017536758753608581,...","([[1.0, 0.47386620792483153, 0.631636308651214..."
5,Shrinkage Estimator,consumer_staples,None,"[[1.0, 0.473865960659252, 0.6316366670257212, ...","[[0.0005528972424157462, 0.0002661029824125156...","[0.0019756257672612805, 0.0017536758753608581,...","([[1.0, 0.473865960659252, 0.6316366670257212,..."
6,MP Distribution,defensive_plays,Dwave,"[[1.0, 0.473865960659252, 0.47184454754186017,...","[[0.0005703533990963529, 0.0002661029824125157...","[0.0017536758753608581, 0.0019756257672612805,...","{'score': -0.0052578247858780375, 'recombined_..."
7,Shrinkage Estimator,defensive_plays,Dwave,"[[1.0, 0.4738657155922928, 0.47184454754186017...","[[0.0005703533990963529, 0.0002661025416857005...","[0.0017536758753608581, 0.001975625375834062, ...","{'score': -0.003785368276536593, 'recombined_s..."
8,MP Distribution,defensive_plays,Gurobi,"[[1.0, 0.473865960659252, 0.47184454754186017,...","[[0.0005703533990963529, 0.0002661029824125157...","[0.0017536758753608581, 0.0019756257672612805,...","{'score': -0.006170034176195616, 'recombined_s..."
9,Shrinkage Estimator,defensive_plays,Gurobi,"[[1.0, 0.4738657155922928, 0.47184454754186017...","[[0.0005703533990963529, 0.0002661025416857005...","[0.0017536758753608581, 0.001975625375834062, ...","{'score': -0.006170034039281824, 'recombined_s..."


In [60]:
# Save the results dataframe to a CSV file
csv_filename = "market_analysis_results.csv"
results_df.to_csv(csv_filename, index=False)
print(f"CSV file saved: {csv_filename}")

CSV file saved: market_analysis_results.csv
